# Sentiment Analysis with ANN

## Installation and Imports

In [ ]:
# Necessary dependencies
!pip install torch sentence-transformers tqdm pandas numpy scikit-learn


In [ ]:
# Necessary imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, classification_report
from sentence_transformers import SentenceTransformer

In [ ]:
# Device-independent code
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Load Datasets

The CSV files should be in a 'data/' directory at the root of the project.

In [ ]:
DEV_FILE = ("data/trac2_CONVT_dev.csv")
TRAIN_FILE = ("data/trac2_CONVT_train.csv")
TEST_FILE = ("data/trac2_CONVT_test.csv")

# Resolved parser error with escaping \" in dialogue columns on designated files
dev_file = pd.read_csv(
    DEV_FILE,
    engine="python",
    escapechar="\\"
)
train_file = pd.read_csv(TRAIN_FILE)
test_file = pd.read_csv(
    TEST_FILE,
    engine="python",
    escapechar="\\"
)

## Preprocessing

In [ ]:
def clean_text(text):
  # Handle NULL value in dataset
  if pd.isna(text):
    return ""
  return text.lower().strip()

dev_file["text"] = dev_file["text"].apply(clean_text)
train_file["text"] = train_file["text"].apply(clean_text)
test_file["text"] = test_file["text"].apply(clean_text)

## Computing Sentence Embeddings

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding dev set")
X_dev = embedder.encode(dev_file["text"].tolist(), show_progress_bar=False)

# Only show progress bar here for screenshot
print("Encoding train set")
X_train = embedder.encode(train_file["text"].tolist(), show_progress_bar=True)

print("Encoding test set")
X_test = embedder.encode(test_file["text"].tolist(), show_progress_bar=False)

# Dividing by 5 to normalize
emotion_dev = dev_file["Emotion"].values.astype(np.float32) / 5.0
empathy_dev = dev_file["Empathy"].values.astype(np.float32) / 5.0
polarity_dev = dev_file["EmotionalPolarity"].values.astype(np.int64)

emotion_train = train_file["Emotion"].values.astype(np.float32) / 5.0
empathy_train = train_file["Empathy"].values.astype(np.float32) / 5.0
polarity_train = train_file["EmotionalPolarity"].values.astype(np.int64)

## Dataset Class

In [ ]:
class SentAnalysisDataset(Dataset):
  def __init__(self, X, emotion=None, empathy=None, polarity=None):
    self.X = torch.tensor(X, dtype=torch.float32)
    self.emotion = emotion
    self.empathy = empathy
    self.polarity = polarity

  def __len__(self):
    return len(self.X)

  # index corresponds to article_id
  def __getitem__(self, index):
    if self.emotion is None:
      return self.X[index]

    return (
        self.X[index],
        torch.tensor(self.emotion[index], dtype=torch.float32),
        torch.tensor(self.empathy[index], dtype=torch.float32),
        torch.tensor(self.polarity[index], dtype=torch.long),
    )

## Dataloaders

In [ ]:
dev_dataset = SentAnalysisDataset(X_dev, emotion_dev, empathy_dev, polarity_dev)
train_dataset = SentAnalysisDataset(X_train, emotion_train, empathy_train, polarity_train)
test_dataset = SentAnalysisDataset(X_test)

dev_loader = DataLoader(dev_dataset, batch_size=32)
train_loader = DataLoader(train_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

## ANN Class

In [ ]:
class SentAnalysisANN(nn.Module):
  def __init__(self, input_dim):
    super().__init__()

    # Layers
    self.fc1 = nn.Linear(input_dim, 256)
    self.fc2 = nn.Linear(256, 128)

    # Heads
    self.emotion_head = nn.Linear(128, 1)
    self.empathy_head = nn.Linear(128, 1)
    self.polarity_head = nn.Linear(128, 4)

    # Dropout
    self.dropout = nn.Dropout(0.2)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = self.dropout(x)
    x = F.relu(self.fc2(x))

    emotion = self.emotion_head(x)
    empathy = self.empathy_head(x)
    polarity = self.polarity_head(x)

    return emotion, empathy, polarity

## Training Setup and Loop

In [ ]:
model = SentAnalysisANN(input_dim=X_train.shape[1]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

mse_loss = nn.MSELoss()
ce_loss = nn.CrossEntropyLoss()

# 80 seems to be where improvement slows down
EPOCHS = 80

for epoch in range(EPOCHS):
  model.train()
  total_loss = 0

  for X, y_emotion, y_empathy, y_polarity in train_loader:
    X = X.to(DEVICE)
    y_emotion = y_emotion.to(DEVICE)
    y_empathy = y_empathy.to(DEVICE)
    y_polarity = y_polarity.to(DEVICE)

    optimizer.zero_grad()

    pred_emotion, pred_empathy, pred_polarity = model(X)

    loss = (
        mse_loss(pred_emotion, y_emotion) +
        mse_loss(pred_empathy, y_empathy) +
        ce_loss(pred_polarity, y_polarity)
    )

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

## Evaluation

In [ ]:
model.eval()

all_preds_emotion = []
all_preds_empathy = []
all_preds_polarity = []

all_true_emotion = []
all_true_empathy = []
all_true_polarity = []

with torch.no_grad():
  for X, y_emotion, y_empathy, y_polarity in dev_loader:
    X = X.to(DEVICE)

    pred_emotion, pred_empathy, pred_polarity = model(X)

    all_preds_emotion.extend(pred_emotion.cpu().numpy())
    all_preds_empathy.extend(pred_empathy.cpu().numpy())
    all_preds_polarity.extend(torch.argmax(pred_polarity, dim=1).cpu().numpy())

    all_true_emotion.extend(y_emotion.numpy())
    all_true_empathy.extend(y_empathy.numpy())
    all_true_polarity.extend(y_polarity.numpy())

all_preds_emotion = np.array(all_preds_emotion) * 5.0
all_preds_empathy = np.array(all_preds_empathy) * 5.0

all_true_emotion = np.array(all_true_emotion) * 5.0
all_true_empathy = np.array(all_true_empathy) * 5.0

print("******** Regression Metrics ********")
print("Emotion Mean Square Error: ", mean_absolute_error(all_true_emotion, all_preds_emotion))
print("Empathy Mean Square Error: ", mean_absolute_error(all_true_empathy, all_preds_empathy))

print("\n******** Classification Metrics ********")
print(classification_report(all_true_polarity, all_preds_polarity))

## Test Model and Save Predictions to File

In [ ]:
model.eval()

preds_test = []

with torch.no_grad():
  for X in test_loader:
    X = X.to(DEVICE)

    pred_emotion, pred_empathy, pred_polarity = model(X)

    emotion = pred_emotion.cpu().numpy() * 5.0
    empathy = pred_empathy.cpu().numpy() * 5.0
    polarity = torch.argmax(pred_polarity, dim=1).cpu().numpy() + 1

    for e, p, em in zip(emotion, polarity, empathy):
      preds_test.append((e, p, em))

out_file = pd.DataFrame({
    "id": test_file["id"],
    "Emotion": [x[0] for x in preds_test],
    "EmotionalPolarity": [x[1] for x in preds_test],
    "Empathy": [x[2] for x in preds_test],
})

out_file.to_csv("predictions_ann.csv")
print("Test predictions saved to predictions_ann.csv")